# Gathering & Exploring FRED Data

This is a brief overview of import FRED data, formatting the data, and creating a visulization of the data.

##Setting up

### libraries

We are installing the fredapi libary first, since this is not already included in the standard colab libraries we need to install it using !pip install fredapi.

Then we need to import all the libraries we will be using.

In [ ]:
!pip install fredapi

In [ ]:
from datetime import date
from fredapi import Fred
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### API Key

You will need to enter your API key here as a the value for the global variable named FRED_API_KEY. You can get an API key from the FRED website.

In [ ]:
FRED_API_KEY = 'enter your api key here'

### Plotting functions

In [ ]:
def data_plot_m1(series, label):
  series.plot(color='cyan', style='-')
  plt.xlabel('Time')
  plt.ylabel('Value')
  plt.title('Time Series')
  plt.legend([label])
  plt.show()

In [ ]:
def data_plot_m2(df, label):
  plt.plot(df.date, df.value)
  plt.xlabel('Time')
  plt.ylabel('Value')
  plt.title('Time Series')
  plt.legend([label])
  plt.show()

## Method 1: Using the fredapi library

### Gathering the data

Using our API we call the fredapi library Fred function, then we use get_series and pass it the series name. We use the pandas head function to look at the first 5 rows of the data.

In [ ]:
fred = Fred(api_key=FRED_API_KEY)
fred_data_m1 = fred.get_series('TTLCONS')
fred_data_m1.head()

### Visualizing the data

Using our function data_plot we pass the series and label.

In [ ]:
data_plot_m1(fred_data_m1, 'Total Const Spending')

## Method 2: Using the API and requests library

###Requesting data

We will create a variable for the base api and a set of parameters as a dictionary per the FRED documentation

In [ ]:
api_url = 'https://api.stlouisfed.org/fred/series/observations'

parameters = {'api_key': FRED_API_KEY,
              'series_id': 'TTLCONS',
              'observation_start': '2013-01-01',
              'observation_end': '2022-01-01',
              'file_type': 'json',
              }

We pass the url and parameters into a new variable, looping through the parameters we joing them with a & symbol to fit the format of the API documentation.

In [ ]:
request_url = api_url + '?' + '&'.join([f'{k}={v}' for k, v in parameters.items()])

Then we request the data and retreive it (get a response) as a json file

In [ ]:
response = requests.get(request_url)
response.raise_for_status()

fred_api_data = response.json()
print(fred_api_data)

We normalize the JSON and create a new data frame, since we only need the date and value columns we drop the realtime start and end columns

In [ ]:
fred_data_m2 = pd.json_normalize(fred_api_data['observations'])

fred_data_m2.head()

In [ ]:
fred_data_m2.drop(fred_data_m2.columns[[0,1]], axis=1, inplace=True)

fred_data_m2.head()

### Converting data types

We first check our data frame data types to see if they are date and numeric values. Since they are not, we create a dictionary with the column names and data types we want to convert to and use the as.type function to convert the data frame data types.

In [ ]:
fred_data_m2.dtypes

In [ ]:
type_convert = {'date': np.datetime64,
                'value': float}

fred_data_m2 = fred_data_m2.astype(type_convert)

fred_data_m2.dtypes

###Plotting the data

Using the function created at the begining we pass our data frame into the function to yield a visualization of our data.

In [ ]:
data_plot_m2(fred_data_m2, 'Total Construction Spending')

## Gathering multiple series

Using the same methods and most of the same code as method 2 we rquest 3 separate data sets from the FRED API.

###Requesting data

To request multiple data sets we create multiple sets of parameters

In [ ]:
api_url = 'https://api.stlouisfed.org/fred/series/observations'

ttl_cons = {'api_key': FRED_API_KEY,
            'series_id': 'TTLCONS',
            'observation_start': '2003-01-01',
            'observation_end': '2023-01-01',
            'file_type': 'json',
            }

ttl_non_res = {'api_key': FRED_API_KEY,
              'series_id': 'TLNRESCONS',
              'observation_start': '2003-01-01',
              'observation_end': '2023-01-01',
              'file_type': 'json',
              }

ttl_res = {'api_key': FRED_API_KEY,
           'series_id': 'TLRESCONS',
           'observation_start': '2003-01-01',
           'observation_end': '2023-01-01',
           'file_type': 'json',
           }

Instead of writing the code over three times for each data frame we create a function that takes the parameters, creates the url, requests the data, gets a response, and processess the data per our previous code.

In [ ]:
def get_fred(parameters, name):
  request_url = api_url + '?' + '&'.join([f'{k}={v}' for k, v in parameters.items()])

  response = requests.get(request_url)
  response.raise_for_status()
  fred_api_data = response.json()

  df = pd.json_normalize(fred_api_data['observations'])
  df.drop(df.columns[[0,1]], axis=1, inplace=True)
  type_convert = {'date': np.datetime64,
                'value': float}
  df = df.astype(type_convert)
  df.rename(columns = {'value' : name}, inplace = True)

  return(df)

Using the fucntion we just created we pass out three sets of paramters and get three data frames in return.

In [ ]:
res = get_fred(ttl_res, 'res')
non_res = get_fred(ttl_non_res, 'nres')
cons = get_fred(ttl_cons, 'cons')

In [ ]:
res.head()

In [ ]:
non_res.head()

In [ ]:
cons.head()

### Merging the data frames

We use the residential data frame as our base and create a new data frame by using the merge function and merging the residnetial and non-residential data frames. We use 'on=' to state that the data frames share and date column, and the 'how=' set to outer to state we want all rows even if they dont exist in both data frames. We repeate this process again to add the total construction data frame.

In [ ]:
subgroups = res.merge(non_res, on='date', how='outer')
all_data = subgroups.merge(cons, on='date', how='outer')
all_data.head()

###Plotting the data

Using matplotlib we give our plot all three data sets in our data frame (by column), label them, create our axises and legend.

In [ ]:
plt.plot(all_data['date'], all_data['res'], label='Residential')
plt.plot(all_data['date'], all_data['nres'], label='Non-Residential')
plt.plot(all_data['date'], all_data['cons'], label='Total')
plt.xlabel('Date')
plt.ylabel('Spending')
plt.title('Total, Residential, and Non-Residential Construction Spending')
_ = plt.legend()